# 위스퍼 모델을 내려받아 로컬에서 사용하기 TH

In [2]:
import os
os.environ["PATH"] += os.pathsep + r"C:\ffmpeg-2025-08-04-git-9a32b86307-full_build\bin"

In [3]:
# 음성을 영어로 번역해서 출력

import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
# from datasets import load_dataset

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3-turbo"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
    return_timestamps=True,   # 청크별로 타임스탬프 반환
    chunk_length_s=10,  # 입력 오디오 10초씩 나누기r
    stride_length_s=2,  # 2초씩 겹치도록 청크 나누기
) 

# dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation")
# sample = dataset[0]["audio"]
sample = "./aug_test.mp3"

result = pipe(sample)
# print(result["text"])

print(result)

Device set to use cpu
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/2868

{'text': " Hello. Today we are going to learn about the language. You will use the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the me

Device set to use cpu
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
{'text': " Hello. Today we are going to learn about the language. ความคาดหวังไว้มากมันอาจจะทำให้เขาอื่นอาจบางครั้งความคาดหวังของคนอื่นมันก็กลายเป็นภาละของเราทั้งหมดนี้ก็เป็นตัวอย่างประโยคที่ใช้คำว่าความคาดหวังค่ะ", 'ความคาดหวังไว้มาก'}, {'timestamp': (20.0, 22.0), 'text': 'มันอาจจะทำให้เขาอื่นอาจ'}, {'timestamp': (22.0, 24.0), 'text': 'บางครั้ง'}, {'timestamp': (24.0, 26.0), 'text': 'ความคาดหวังของคนอื่น'}, {'timestamp': (26.0, 28.0), 'text': 'มันก็กลายเป็น'}, {'timestamp': (28.0, 30.0), 'text': 'ภาละของเรา'}, {'timestamp': (30.0, 34.7), 'text': 'ทั้งหมดนี้ก็เป็นตัวอย่างประโยคที่ใช้คำว่าความคาดหวังค่ะ'}]}

model="Helsinki-NLP/opus-mt-tc-big-en-ko",  # 영어->한국어 번역 모델  
model="Helsinki-NLP/opus-mt-ko-en",  # 한국어 -> 영어 번역 모델  
model="Helsinki-NLP/opus-mt-th-en",  # 태국어->영어 번역 모델

In [12]:
from transformers import pipeline

# 음성 인식 파이프라인 (원본 언어 텍스트 추출)
asr_pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
    return_timestamps=True,
    chunk_length_s=10,
    stride_length_s=2
)

# 태국어 -> 영어 번역 파이프라인
translation_th_en = pipeline(
    "translation", 
    model="Helsinki-NLP/opus-mt-th-en",  # 태국어->영어 번역 모델
    device=device
)

# 영어 -> 한국어 번역 파이프라인
translation_en_ko = pipeline(
    "translation", 
    model="Helsinki-NLP/opus-mt-tc-big-en-ko",  # 영어->한국어 번역 모델
    device=device
)

# 영어 -> 한국어 번역 파이프라인
translation_en_ko = pipeline(
    "translation", 
    model="Helsinki-NLP/opus-mt-ko-en",  # 한국어 - 영어 번역 모델
    device=device
)



# 샘플 오디오 파일 경로
sample = "./aug_test_cut.mp3"

# 음성 인식 (원본 언어 텍스트 추출)
asr_result = asr_pipe(sample)

# 원본 텍스트와 번역 텍스트 출력
original_text = asr_result['text']

# 태국어 텍스트를 영어로 번역
translation_th_en_result = translation_th_en(original_text)

# 영어 텍스트를 한국어로 번역
translation_en_ko_result = translation_en_ko(translation_th_en_result[0]['translation_text'])

# 최종 번역된 한국어 텍스트
translated_text = translation_en_ko_result[0]['translation_text']

# 원본 텍스트와 번역 텍스트 출력 (줄바꿈으로 나누기)
print(f"원본 텍스트:\n{original_text}\n")
print(f"한국어 번역 텍스트:\n{translated_text}")


Device set to use cpu
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
c:\Users\pc04-06\anaconda3\envs\my_llm\Lib\site-packages\transformers\models\marian\tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use cpu
Device set to use cpu
Device set to use cpu


원본 텍스트:
 ความข้าดหวัง เดี๋ยวคุณคูจะยกตัวอย่างประโยค ที่ใช้คำว่าความข้าดหวังค่ะ บางทีความข้ดหวังมันก็อาจจะทำให้เราเหนื่อยโดยที่เราไม่รู้ตัว

한국어 번역 텍스트:
I'm finding that you're giving us an example of that saying "happily," and maybe it'll get us tired without knowing it.


In [17]:
# 음성 인식 결과 확인
print(f"원본 태국어 텍스트:\n{original_text}")


원본 태국어 텍스트:
 ความข้าดหวัง เดี๋ยวคุณคูจะยกตัวอย่างประโยค ที่ใช้คำว่าความข้าดหวังค่ะ บางทีความข้ดหวังมันก็อาจจะทำให้เราเหนื่อยโดยที่เราไม่รู้ตัว


In [19]:
# 다중 언어 번역 모델 다른걸로 실행 facebook

from transformers import pipeline
import torch

# 음성 인식 파이프라인 (원본 언어 텍스트 추출)
asr_pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
    return_timestamps=True,
    chunk_length_s=10,
    stride_length_s=2
)

# 다국어 번역 모델 (facebook/m2m100_418M 사용)
translation_th_ko = pipeline(
    "translation", 
    model="facebook/m2m100_418M",  # 다국어 번역 모델
    device=device,
    src_lang="th",  # 태국어 (출발 언어)
    tgt_lang="ko"   # 한국어 (목표 언어)
)

# 샘플 오디오 파일 경로
sample = "./aug_test_cut.mp3"

# 음성 인식 (원본 언어 텍스트 추출)
asr_result = asr_pipe(sample)

# 원본 태국어 텍스트
original_text = asr_result['text']
print(f"원본 태국어 텍스트:\n{original_text}")

# 태국어 텍스트를 한국어로 번역
translation_th_ko_result = translation_th_ko(original_text)
translated_text = translation_th_ko_result[0]['translation_text']

# 최종 번역된 한국어 텍스트 출력
print(f"최종 한국어 번역 텍스트:\n{translated_text}")


Device set to use cpu
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Device set to use cpu


원본 태국어 텍스트:
 ความข้าดหวัง เดี๋ยวคุณคูจะยกตัวอย่างประโยค ที่ใช้คำว่าความข้าดหวังค่ะ บางทีความข้ดหวังมันก็อาจจะทำให้เราเหนื่อยโดยที่เราไม่รู้ตัว
최종 한국어 번역 텍스트:
희망은 희망이라는 단어를 사용합니다. 어쩌면 희망은 우리가 알지 못함으로써 우리를 피곤하게 만들 수 있습니다.


In [4]:
# 원본 태국어 텍스트 추출 후, 두 가지 번역 모델 (facebook/m2m100_418M과 Helsinki-NLP/opus-mt-th-en)을 사용하여 태국어 -> 영어 번역 결과를 각각 출력



import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
# from datasets import load_dataset

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3-turbo"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

# 음성 인식 파이프라인 (원본 언어 텍스트 추출)
asr_pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
    return_timestamps=True,
    chunk_length_s=10,
    stride_length_s=2
)

# 태국어 -> 영어 번역 파이프라인 (Helsinki-NLP/opus-mt-th-en)
translation_th_en_1 = pipeline(
    "translation", 
    model="Helsinki-NLP/opus-mt-th-en",  # 태국어->영어 번역 모델
    device=device
)

# 태국어 -> 영어 번역 파이프라인 (facebook/m2m100_418M)
translation_th_en_2 = pipeline(
    "translation", 
    model="facebook/m2m100_418M",  # 다국어 모델 (태국어 -> 영어)
    device=device,
    src_lang="th",  # 태국어 (출발 언어)
    tgt_lang="en"   # 영어 (목표 언어)
)

# 샘플 오디오 파일 경로
sample = "./aug_test_cut.mp3"

# 음성 인식 (원본 언어 텍스트 추출)
asr_result = asr_pipe(sample)

# 원본 태국어 텍스트
original_text = asr_result['text']
print(f"원본 태국어 텍스트:\n{original_text}")

# 1. Helsinki-NLP/opus-mt-th-en 모델을 사용하여 번역
translation_th_en_result_1 = translation_th_en_1(original_text)
translated_text_1 = translation_th_en_result_1[0]['translation_text']
print(f"\nHelsinki-NLP/opus-mt-th-en 모델 번역:\n{translated_text_1}")

# 2. facebook/m2m100_418M 모델을 사용하여 번역
translation_th_en_result_2 = translation_th_en_2(original_text)
translated_text_2 = translation_th_en_result_2[0]['translation_text']
print(f"\nfacebook/m2m100_418M 모델 번역:\n{translated_text_2}")


Device set to use cpu
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
c:\Users\pc04-06\anaconda3\envs\my_llm\Lib\site-packages\transformers\models\marian\tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use cpu
Device set to use cpu
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection follo

원본 태국어 텍스트:
 ความข้าดหวัง เดี๋ยวคุณคูจะยกตัวอย่างประโยค ที่ใช้คำว่าความข้าดหวังค่ะ บางทีความข้ดหวังมันก็อาจจะทำให้เราเหนื่อยโดยที่เราไม่รู้ตัว

Helsinki-NLP/opus-mt-th-en 모델 번역:
I'm hoping that you'll give us an example of a sentence that says "hopeful," and maybe it'll get us tired without knowing it.

facebook/m2m100_418M 모델 번역:
Hopefully, you’re going to raise an example. Use the word hopefully. Maybe hopefully, it might get us tired because we don’t know it.


원본 태국어 텍스트:
 ความข้าดหวัง เดี๋ยวคุณคูจะยกตัวอย่างประโยค ที่ใช้คำว่าความข้าดหวังค่ะ บางทีความข้ดหวังมันก็อาจจะทำให้เราเหนื่อยโดยที่เราไม่รู้ตัว

Helsinki-NLP/opus-mt-th-en 모델 번역:
I'm hoping that you'll give us an example of a sentence that says "hopeful," and maybe it'll get us tired without knowing it.

facebook/m2m100_418M 모델 번역:
Hopefully, you’re going to raise an example. Use the word hopefully. Maybe hopefully, it might get us tired because we don’t know it.

In [ ]:
# 원본 태국어 텍스트 추출 후, 두 가지 번역 모델 (facebook/m2m100_418M과 Helsinki-NLP/opus-mt-th-en)을 사용하여 태국어 -> 영어 번역 결과를 각각 출력



import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
# from datasets import load_dataset

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3-turbo"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

# 음성 인식 파이프라인 (원본 언어 텍스트 추출)
asr_pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
    return_timestamps=True,
    chunk_length_s=10,
    stride_length_s=2
)

# 태국어 -> 영어 번역 파이프라인 (Helsinki-NLP/opus-mt-th-en)
translation_th_en_1 = pipeline(
    "translation", 
    model="Helsinki-NLP/opus-mt-th-en",  # 태국어->영어 번역 모델
    device=device
)

# 태국어 -> 영어 번역 파이프라인 (facebook/m2m100_418M)
translation_th_en_2 = pipeline(
    "translation", 
    model="facebook/m2m100_418M",  # 다국어 모델 (태국어 -> 영어)
    device=device,
    src_lang="th",  # 태국어 (출발 언어)
    tgt_lang="en"   # 영어 (목표 언어)
)

# 샘플 오디오 파일 경로
sample = "./aug_test.mp3"

# 음성 인식 (원본 언어 텍스트 추출)
asr_result = asr_pipe(sample)

# 원본 태국어 텍스트
original_text = asr_result['text']
print(f"원본 태국어 텍스트:\n{original_text}")

# 1. Helsinki-NLP/opus-mt-th-en 모델을 사용하여 번역
translation_th_en_result_1 = translation_th_en_1(original_text)
translated_text_1 = translation_th_en_result_1[0]['translation_text']
print(f"\nHelsinki-NLP/opus-mt-th-en 모델 번역:\n{translated_text_1}")

# 2. facebook/m2m100_418M 모델을 사용하여 번역
translation_th_en_result_2 = translation_th_en_2(original_text)
translated_text_2 = translation_th_en_result_2[0]['translation_text']
print(f"\nfacebook/m2m100_418M 모델 번역:\n{translated_text_2}")


# 3. Helsinki-NLP/opus-mt-th-en 모델을 사용하여 번역


# 4. facebook/m2m100_418M 모델을 사용하여 번역

translation_en_ko_result = translation_en_ko(original_text)
translated_text = translation_en_ko_result[0]['translation_text']


Device set to use cpu
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Device set to use cpu
Your input_length: 525 is bigger than 0.9 * max_length: 200. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)


원본 태국어 텍스트:
 Hello. Today we are going to learn about the language. You will use the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the meaning of the 

Device set to use cpu
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Device set to use cpu
Your input_length: 525 is bigger than 0.9 * max_length: 200. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)
원본 태국어 텍스트:
 Hello. Today we are going to learn about the language. You will use the meaning of the It may make you nervous when you don't know. The way we make you nervous, it may make you nervous.ความคาดหวังไว้มากมันอาจจะทำให้เขาอื่นอาจบางครั้งความคาดหวังของคนอื่นมันก็กลายเป็นภาละของเราทั้งหมดนี้ก็เป็นตัวอย่างประโยคที่ใช้คำว่าความคาดหวังค่ะ
최종 한국어 번역 텍스트:
오늘 우리는 언어에 대해 배울 것입니다. 당신은 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미의 의미

In [15]:
# chunks를 CSV 파일로 저장
start_end_text = []

for chunk in result["chunks"]:
    start = chunk["timestamp"][0]
    end = chunk["timestamp"][1]
    text = chunk["text"]
    start_end_text.append([start, end, text])

import pandas as pd
df = pd.DataFrame(start_end_text, columns=["start", "end", "text"])
df.to_csv("aug_test_cut.csv", index=False, sep="|")
display(df)

,start,end,text
0,0.0,5.0,Hello. Today we are going to learn about the ...
1,5.0,16.0,You will use the meaning of the meaning of th...
2,16.0,20.0,"The way we make you nervous, it may make you ..."
3,20.0,22.0,มันอาจจะทำให้เขาอื่นอาจ
4,22.0,24.0,บางครั้ง
5,24.0,26.0,ความคาดหวังของคนอื่น
6,26.0,28.0,มันก็กลายเป็น
7,28.0,30.0,ภาละของเรา
8,30.0,34.7,ทั้งหมดนี้ก็เป็นตัวอย่างประโยคที่ใช้คำว่าความค...
